# BIPO steering: multi-turn (temporal) run — pilot

Pilot for the long-horizon experiment for BIPO steering, with the same protocol as
`scripts/direct_prompting_temporal.py` (direct prompting) and `scripts/rewriting_temporal.py`
(post-generation rewriting), so the three methods can be compared item by item:

- the same targets (`sample_ids`, seed 20261026) from the BehaviorBench test split
- the same history order (`batches_ending_with_id`), up to 10 turns
- probes at list lengths 5 and 10: the target is asked on a copy of the conversation and recorded, not appended
- free-text answers (`max_new_tokens=128`, greedy), then blind extraction and `score_extraction`

The only difference is how the target is set. There is no persona system prompt and no rewriter.
Every turn uses the unconditioned draft prompt (`prompts.draft_messages`), and the trait's BIPO
vector is added at layer 15 with coefficient `score_to_coeff(level, k) = k (level - 3) / 2`.
The vectors, layer and k = 4 are the ones from the single-turn run in
`evaluate_bipo_behaviorbench.ipynb`. Extraction is unsteered.

The pilot matches the other two methods' pilots: 2 targets per trait, levels 1 and 5,
10 turns, probes at 5 and 10 (20 conversations, 40 records). Steering adds one coefficient to
the whole batch, so batches are grouped by (trait, level).

## Setup
Run from the repo (or `notebooks/`). Loads 4-bit NF4 on CUDA, as in the other temporal runs; otherwise the unquantized fallback (fp16 on MPS), as in the single-turn BIPO run.

In [1]:
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
for path in (ROOT, ROOT / 'src', ROOT / 'scripts'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import bipo_temporal
from rewriting.llm import HFGenerator

generator = HFGenerator.load(load_4bit=torch.cuda.is_available())
vectors = bipo_temporal.load_vectors()
parameter = next(generator.model.parameters())
print('device', parameter.device, '| dtype', parameter.dtype, '| 4-bit', torch.cuda.is_available())
print('layer', bipo_temporal.LAYER, '| k', bipo_temporal.K,
      '| vector norms', {t: round(float(v.norm()), 3) for t, v in vectors.items()})

OUT = ROOT / 'outputs' / 'bipo'   # gitignored

W1124 05:40:49.416000 78455 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

device mps:0 | dtype torch.float16 | 4-bit False
layer 15 | k 4.0 | vector norms {'O': 0.74, 'C': 0.629, 'E': 0.659, 'A': 0.472, 'N': 0.542}


## Pilot
2 targets per trait at levels 1 and 5 (20 conversations, 40 records). Each list length prints a line as it finishes.

In [2]:
import time

start = time.time()
bipo_temporal.run_experiment(generator, OUT / 'temporal_pilot.jsonl', ids_per_trait=2, levels=[1, 5],
                             max_turns=10, probe_turns=[5, 10], batch_size=8, vectors=vectors)
pilot_minutes = (time.time() - start) / 60
print(f'pilot: {pilot_minutes:.1f} min')

  list length 1: 20 conversations


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  list length 2: 20 conversations
  list length 3: 20 conversations
  list length 4: 20 conversations
  list length 5: 20 conversations
  list length 6: 20 conversations
  list length 7: 20 conversations
  list length 8: 20 conversations
  list length 9: 20 conversations
  list length 10: 20 conversations
seed 20261026: 40 records
pilot: 33.5 min


## Pilot check
Invalid and refusal rates, endorsement by level, and the answers themselves. Level 1 and level 5 should read differently and score differently on most traits.

In [3]:
import pandas as pd

pd.set_option('display.max_colwidth', 220)
pilot = pd.read_json(OUT / 'temporal_pilot.jsonl', lines=True)
TURNS = {4: 'Turn 5', 9: 'Turn 10'}
print(len(pilot), 'records | invalid', pilot.score.isna().mean().round(3), '| refusal', pilot.refusal.mean().round(3),
      '| empty answers', (pilot.text.str.len() == 0).sum())
print(pilot.pivot_table(index='target_trait', columns=['turn', 'level'], values='score', aggfunc='mean')
      .rename(columns=TURNS, level=0))

40 records | invalid 0.0 | refusal 0.0 | empty answers 0
turn         Turn 5      Turn 10     
level             1    5       1    5
target_trait                         
A               1.0  1.0     1.0  1.0
C               1.0  1.0     1.0  1.0
E               1.0  1.0     0.5  0.5
N               1.0  1.0     1.0  1.0
O               1.0  1.0     0.5  1.0


In [4]:
# The same target at level 1 and level 5, after 10 turns
pilot[pilot.turn == 9].sort_values(['target_trait', 'item_id', 'level'])[
    ['target_trait', 'item_id', 'level', 'coeff', 'text', 'extracted', 'score']]

,target_trait,item_id,level,coeff,text,extracted,score
29,A,bb-A-004-03,1,-4,"I would hold the elevator door open for my neighbor, as it's a minor inconvenience to delay my exit and it's a common courtesy to let someone who is approaching the elevator have a chance to get on. Holding the door ...",Yes,1
31,A,bb-A-004-03,5,4,"I'd hold the door open for my neighbor, as it's a warm and welcoming gesture that shows I value our friendship and community connection. We often exchange smiles and hellos, and this small act feels like a way to con...",Yes,1
25,A,bb-A-004-06,1,-4,"I would hold the door open for the acquaintance, as it's a minor inconvenience to delay the elevator and it's a common courtesy to let someone who is clearly in a hurry catch up. Holding the door open is a simple and...",Yes,1
27,A,bb-A-004-06,5,4,"I'd hold the door open for my friend, as it's a warm and welcoming gesture that shows I care about their well-being. We've built a friendly connection at the gym, and this small act of kindness feels like a way to nu...",Yes,1
9,C,bb-C-003-08,1,-4,"I'd keep everything neat by throwing away the expired containers. It's a good idea to clean up after yourself and others, and getting rid of the old containers will help keep the fridge from getting messy and gross. ...",Yes,1
11,C,bb-C-003-08,5,4,"I would remove the expired containers and leave the fridge in pristine condition. This is because I recognize that maintaining a clean and organized shared space is a collective responsibility, and removing the expir...",Yes,1
13,C,bb-C-008-02,1,-4,"I'd try to stay calm and not get too worked up about it. It's okay if people have different opinions, and it's not worth getting defensive or emotional about a social media post. I'd just scroll past the post and mov...",No,1
15,C,bb-C-008-02,5,4,"I would not get carried away by feelings. This is because I recognize that social media can be a breeding ground for misinformation, and I have a responsibility to critically evaluate the information presented, rathe...",No,1
17,E,bb-E-001-05,1,-4,"I would not give the video a thumbs up, as I'm not actively seeking information about stretching techniques and the video is simply a passive suggestion that I'm not obligated to engage with. I don't feel that giving...",No,0
19,E,bb-E-001-05,5,4,"I'm swiping left on that video - I just finished a killer workout and I'm not here to give some random user a thumbs up for their stretching skills. I'm already on the app to get pumped up and motivated, not to give ...",No,0


In [5]:
# Invalid answers: what the extractor saw
pilot[pilot.score.isna()][['target_trait', 'level', 'turn', 'item_id', 'text', 'extracted']]

,target_trait,level,turn,item_id,text,extracted


In [6]:
# Rough time for the full run: 250 conversations at batch_size 16 in groups of 10 per (trait, level),
# so about 25 calls per step instead of the pilot's 10 (groups of 2). Contexts are the same length.
print(f'full run estimate: ~{pilot_minutes * 25 / 10 * 1.3:.0f} min (very rough)')

full run estimate: ~109 min (very rough)


## Full run

All 50 targets (10 per trait), levels 1-5, 10 turns: 250 conversations, `batch_size=16`, as in the
direct prompting and rewriting runs. Level 3 has coefficient 0, so it is the unsteered conversation.

The target is also probed at list length 1 (no history). Probes never enter the history, so this
leaves the turn 5 and turn 10 records unchanged. It gives a free-text single-turn reference on the
same 50 items, because the single-turn BIPO run asked for a one-word Yes/No (or A/B) answer.
750 records.

In [7]:
import time

FULL = ROOT / 'artifacts' / 'steering' / 'bipo' / 'temporal.jsonl'

start = time.time()
bipo_temporal.run_experiment(generator, FULL, max_turns=10, probe_turns=[1, 5, 10],
                             levels=[1, 2, 3, 4, 5], batch_size=16, vectors=vectors)
print(f'full run: {(time.time() - start) / 60:.1f} min')

  list length 1: 250 conversations
  list length 2: 250 conversations
  list length 3: 250 conversations
  list length 4: 250 conversations
  list length 5: 250 conversations
  list length 6: 250 conversations
  list length 7: 250 conversations
  list length 8: 250 conversations
  list length 9: 250 conversations
  list length 10: 250 conversations
seed 20261026: 750 records
full run: 189.6 min


## Results

Same measures as the rewriting summary: the endorsement slope across levels 1-5 (least squares on the
per-level means), with a 95% bootstrap CI over the 10 target items per trait (1,000 resamples).

In [8]:
import numpy as np
import pandas as pd

df = pd.read_json(FULL, lines=True)
TURNS = {0: 'Turn 1', 4: 'Turn 5', 9: 'Turn 10'}
NAMES = dict(O='openness', C='conscientiousness', E='extraversion', A='agreeableness', N='neuroticism')
LEVELS = np.arange(1, 6)
print(len(df), 'records | refusal', df.refusal.mean().round(3), '| invalid', df.score.isna().mean().round(3),
      '| empty answers', (df.text.str.len() == 0).sum())
print('\ninvalid answers by turn and trait')
print(df.groupby(['turn', 'target_trait']).score.apply(lambda s: s.isna().sum()).unstack().rename(index=TURNS))

for turn, name in TURNS.items():
    print(f'\nendorsement, {name} (trait x level)')
    print(df[df.turn == turn].pivot_table(index='target_trait', columns='level', values='score',
                                          aggfunc='mean').round(2))

750 records | refusal 0.0 | invalid 0.001 | empty answers 0

invalid answers by turn and trait
target_trait  A  C  E  N  O
turn                       
Turn 1        0  0  0  1  0
Turn 5        0  0  0  0  0
Turn 10       0  0  0  0  0

endorsement, Turn 1 (trait x level)
level            1    2    3    4    5
target_trait                          
A             1.00  1.0  1.0  1.0  1.0
C             0.70  0.6  0.9  0.9  1.0
E             0.80  0.8  0.9  1.0  1.0
N             0.89  0.9  0.9  0.9  0.8
O             0.60  0.6  0.6  0.6  0.7

endorsement, Turn 5 (trait x level)
level           1    2    3    4    5
target_trait                         
A             1.0  1.0  1.0  1.0  1.0
C             0.8  0.8  0.9  0.9  0.9
E             0.8  1.0  1.0  1.0  1.0
N             1.0  1.0  1.0  1.0  1.0
O             0.7  0.7  0.7  0.6  1.0

endorsement, Turn 10 (trait x level)
level           1    2    3    4    5
target_trait                         
A             1.0  1.0  1.0  1.0  1.0


In [9]:
def slope(group):
    means = group.groupby('level').score.mean().reindex(LEVELS)
    ok = means.notna().values
    return np.polyfit(LEVELS[ok], means.values[ok], 1)[0]


def item_bootstrap(group, n=1000, seed=20261026):
    rng = np.random.default_rng(seed)
    by_item = {item: rows for item, rows in group.groupby('item_id')}
    items = list(by_item)
    draws = [slope(pd.concat([by_item[i] for i in rng.choice(items, len(items))])) for _ in range(n)]
    return np.nanpercentile(draws, [2.5, 97.5])


# Single-turn reference: the BIPO test run (all 30/50 test items per trait, one-word answers).
single = pd.read_csv(ROOT / 'artifacts' / 'steering' / 'bipo' / 'behaviorbench_items.csv')
single_slope = {code: np.polyfit(LEVELS, single[single.trait == name].groupby('target_score').score.mean()
                                 .reindex(LEVELS).values, 1)[0] for code, name in NAMES.items()}

rows = []
for trait in 'OCEAN':
    row = {'Trait': trait, 'Single-turn': f"{single_slope[trait]:+.2f}"}
    for turn, name in TURNS.items():
        group = df[(df.turn == turn) & (df.target_trait == trait)]
        low, high = item_bootstrap(group)
        row[name] = f"{slope(group):+.2f} [{low:.2f}, {high:.2f}]"
    rows.append(row)
summary = pd.DataFrame(rows).set_index('Trait')
summary

,Single-turn,Turn 1,Turn 5,Turn 10
Trait,,,,
O,+0.05,"+0.02 [-0.00, 0.06]","+0.05 [-0.01, 0.11]","+0.10 [0.04, 0.17]"
C,-0.01,"+0.09 [0.02, 0.16]","+0.03 [-0.04, 0.11]","+0.02 [-0.00, 0.06]"
E,+0.02,"+0.06 [-0.00, 0.15]","+0.04 [-0.00, 0.10]","+0.12 [0.04, 0.20]"
A,+0.15,"-0.00 [-0.00, -0.00]","-0.00 [-0.00, -0.00]","-0.00 [-0.00, -0.00]"
N,+0.02,"-0.02 [-0.06, -0.00]","-0.00 [-0.00, -0.00]","-0.00 [-0.00, -0.00]"


In [10]:
# Low pole vs high pole: shift from the unsteered conversation (level 3, coefficient 0).
# The pilot suggested -4 does much less than +4.
means = df.pivot_table(index=['turn', 'target_trait'], columns='level', values='score', aggfunc='mean')
poles = pd.DataFrame({'level 1 - level 3': means[1] - means[3], 'level 5 - level 3': means[5] - means[3]})
poles.rename(index=TURNS, level=0).round(2).unstack(0)

level 1 - level 3                level 5 - level 3               
turn                    Turn 1 Turn 10 Turn 5            Turn 1 Turn 10 Turn 5
target_trait                                                                  
A                         0.00     0.0    0.0               0.0     0.0    0.0
C                        -0.20    -0.1   -0.1               0.1     0.0    0.0
E                        -0.10    -0.3   -0.2               0.1     0.2    0.0
N                        -0.01     0.0    0.0              -0.1     0.0    0.0
O                         0.00     0.0    0.0               0.1     0.5    0.3

In [11]:
# Does the answer to the same item change between turns (same conversation)?
wide = df.pivot_table(index=['conv_id', 'target_trait', 'level'], columns='turn', values='score')
both = wide.dropna(subset=[4, 9])
changed = both[4] != both[9]
print(f'changed between turn 5 and turn 10: {changed.sum()} of {len(both)} ({changed.mean():.0%})')
print(changed.groupby(level='target_trait').agg(['sum', 'count']))
both_1 = wide.dropna(subset=[0, 9])
print(f'changed between turn 1 and turn 10: {(both_1[0] != both_1[9]).sum()} of {len(both_1)}')

print('\nendorsement at the extreme levels by turn')
print(df[df.level.isin([1, 5])].pivot_table(index='target_trait', columns=['level', 'turn'],
                                            values='score', aggfunc='mean').rename(columns=TURNS, level=1).round(2))

changed between turn 5 and turn 10: 27 of 250 (11%)
              sum  count
target_trait            
A               0     50
C               6     50
E              16     50
N               0     50
O               5     50
changed between turn 1 and turn 10: 36 of 249

endorsement at the extreme levels by turn
level             1                     5               
turn         Turn 1 Turn 5 Turn 10 Turn 1 Turn 5 Turn 10
target_trait                                            
A              1.00    1.0     1.0    1.0    1.0     1.0
C              0.70    0.8     0.9    1.0    0.9     1.0
E              0.80    0.8     0.4    1.0    1.0     0.9
N              0.89    1.0     1.0    0.8    1.0     1.0
O              0.60    0.7     0.5    0.7    1.0     1.0


In [12]:
# Is the steering visible in the text? Levels 1, 3 and 5 on the same target, after 10 turns.
pd.set_option('display.max_colwidth', 220)
at = df[(df.turn == 9) & df.level.isin([1, 3, 5])]
first_items = at.groupby('target_trait').item_id.unique().map(lambda ids: ids[:2]).explode()
at[at.item_id.isin(first_items)].sort_values(['target_trait', 'item_id', 'level'])[
    ['target_trait', 'item_id', 'level', 'text', 'extracted', 'score']]

,target_trait,item_id,level,text,extracted,score
452,A,bb-A-002-07,1,"I would give the former roommate $40, as they are in a difficult situation and I have the means to help. Refusing to help someone in need, especially a former roommate, could damage our relationship and my reputation...",Yes,1.0
458,A,bb-A-002-07,3,"I would give my former roommate the $40. I consider them a friend and am willing to help them in their time of need, especially since they're in a situation where they need access to essential medication. Giving them...",Yes,1.0
464,A,bb-A-002-07,5,"I would give my former roommate the $40, as it's a small act of kindness that can help them access the medication they need. I care about their well-being and feel a connection from our shared living experience, maki...",Yes,1.0
467,A,bb-A-004-06,1,"I would hold the door open for the acquaintance, as it's a minor inconvenience to delay the elevator and it's a common courtesy to let someone who is clearly in a hurry catch up. Holding the door open is a simple and...",Yes,1.0
473,A,bb-A-004-06,3,"I would hold the door open for my acquaintance. I consider them a friend, and holding the door open is a small act of kindness that shows I value our relationship and am willing to go out of my way to help them. It's...",Yes,1.0
479,A,bb-A-004-06,5,"I'd hold the door open for my friend, as it's a warm and welcoming gesture that shows I care about their well-being. We've built a friendly connection at the gym, and this small act of kindness feels like a way to nu...",Yes,1.0
167,C,bb-C-004-03,1,"I'd persist despite the setbacks and try to fix the furniture. It's frustrating to have to disassemble and reassemble parts, but it's better to take the time to do it right than to leave the furniture assembled incor...",Yes,1.0
173,C,bb-C-004-03,3,"I would persist despite the setbacks. I've already invested time and effort into assembling the furniture, and I'm not willing to give up now. By persisting, I can fix the mistake and complete the project, which will...",Yes,1.0
179,C,bb-C-004-03,5,I would persist despite setbacks and immediately disassemble the three sections to correct the critical part's installation. This decision is driven by a commitment to quality and a recognition that the initial mista...,Yes,1.0
152,C,bb-C-008-07,1,"I'd try to stay calm and not get too worked up about the call. It's just a game, and getting too emotional can make it harder to enjoy the rest of the match. I'd rather focus on the fun of watching the game with frie...",No,1.0


In [13]:
# Invalid answers (score is None): what the extractor saw
df[df.score.isna()][['target_trait', 'level', 'turn', 'item_id', 'text', 'extracted']].head(20)

,target_trait,level,turn,item_id,text,extracted
675,N,1,0,bb-N-074-01,"I'd strike up a conversation with the person, as it's a casual and relaxed atmosphere at the coffee shop. I'm curious about their interests and think it's a great opportunity to meet someone with a similar passion fo...",Unclear


In [14]:
# BIPO vs rewriting (artifacts/rewriting/temporal.jsonl): same targets, history order and probes.
rw = pd.read_json(ROOT / 'artifacts' / 'rewriting' / 'temporal.jsonl', lines=True)
pd.DataFrame({(method, TURNS[turn]): {t: slope(d[(d.turn == turn) & (d.target_trait == t)]) for t in 'OCEAN'}
              for method, d in (('bipo', df), ('rewriting', rw)) for turn in (4, 9)}).round(2)

bipo         rewriting        
  Turn 5 Turn 10    Turn 5 Turn 10
O   0.05    0.10      0.06    0.10
C   0.03    0.02      0.19    0.05
E   0.04    0.12      0.24    0.24
A  -0.00   -0.00      0.10    0.13
N  -0.00   -0.00     -0.06   -0.05

## Summary table
Renders the table in the same format as the rewriting summary, plus the counts used in the bullets.

In [15]:
from IPython.display import Markdown

invalid = df[df.score.isna()]
both = wide.dropna(subset=[4, 9])
n_changed = int((both[4] != both[9]).sum())

lines = ['| Trait | Single-turn | Turn 1 | Turn 5 | Turn 10 |', '|---|---|---|---|---|']
lines += [f"| {t} | {r['Single-turn']} | {r['Turn 1']} | {r['Turn 5']} | {r['Turn 10']} |"
          for t, r in summary.iterrows()]
lines = lines[:2] + [line.replace('-', '−') for line in lines[2:]]
by_trait = ', '.join(f'{t} {n}' for t, n in invalid.target_trait.value_counts().items()) or 'none'
facts = [
    f'- Invalid answers: {len(invalid)} of {len(df)} ({by_trait}).',
    f'- The answer to the same item changes between turn 5 and turn 10 in {n_changed} of {len(both)} '
    f'conversations ({n_changed / len(both):.0%}).',
]
Markdown('\n'.join(lines) + '\n\n' + '\n'.join(facts))

| Trait | Single-turn | Turn 1 | Turn 5 | Turn 10 |
|---|---|---|---|---|
| O | +0.05 | +0.02 [−0.00, 0.06] | +0.05 [−0.01, 0.11] | +0.10 [0.04, 0.17] |
| C | −0.01 | +0.09 [0.02, 0.16] | +0.03 [−0.04, 0.11] | +0.02 [−0.00, 0.06] |
| E | +0.02 | +0.06 [−0.00, 0.15] | +0.04 [−0.00, 0.10] | +0.12 [0.04, 0.20] |
| A | +0.15 | −0.00 [−0.00, −0.00] | −0.00 [−0.00, −0.00] | −0.00 [−0.00, −0.00] |
| N | +0.02 | −0.02 [−0.06, −0.00] | −0.00 [−0.00, −0.00] | −0.00 [−0.00, −0.00] |

- Invalid answers: 1 of 750 (N 1).
- The answer to the same item changes between turn 5 and turn 10 in 27 of 250 conversations (11%).

## Multi-turn BIPO steering

Runs the long-horizon (temporal) experiment for BIPO steering, matching the direct prompting setup in
`direct_prompting_temporal.py` and the rewriting setup in `rewriting_temporal.py` so the methods can be
compared item by item. Vectors, layer 15 and k = 4 are from the single-turn BIPO run.

### Results (endorsement slope across levels 1–5, 95% item-bootstrap CI)

_Table from the cell above, filled in after the run._

Single-turn is the single-turn BIPO test run (one-word answers, all test items). Turn 1 is the free-text
answer with no history on the 50 temporal targets.

- _Findings, written after the run._